# RMIT Hackathon 2026 · Challenge 1: Multilingual Prompt-Attack Classification

**Model: FacebookAI/xlm-roberta-base**

**Team:** ShuijiaoT07 · **Institution:** Taylor's University Malaysia

**Task:** classify each prompt as `BENIGN`, `JAILBREAK` or `PROMPT_INJECTION` (pt 45%, fr 25%, it 20%, en 10%).
**Metric:** macro-averaged F1, so the rare `PROMPT_INJECTION` class (7%) counts as much as `BENIGN` (62%).

### Approach at a glance
- **Backbone:** `FacebookAI/xlm-roberta-base` (approved), fine-tuned end-to-end with `AutoModelForSequenceClassification` (3 labels).
- **Validation:** stratified on *label × language*. `fast` mode uses one 10% holdout; `kfold` mode uses 5 folds with out-of-fold (OOF) predictions.
- **Imbalance:** class-weighted cross-entropy (inverse-sqrt frequency) in a custom `Trainer`.
- **Long prompts:** token-level **head + tail truncation** (first 60% / last 40% of the budget), because injected instructions often sit at the end of a long prompt.
- **Decision rule:** per-class additive **logit biases** tuned on validation/OOF to maximise macro-F1 (smoothed grid search, guards against overfitting), then applied unchanged to test.
- **Checks:** per-language macro-F1, confusion matrices before and after tuning, a strict submission validator, and per-stage timing.
- **Reproducible:** fixed seeds, deterministic cuDNN, no internet, no extra installs. The only output is `/kaggle/working/Taylor_MYS_ShuijiaoT07_C1.csv`.

## 0 · Configuration
This is the **only cell you should need to edit**. Defaults: `xlm-roberta-base`, `fast` mode, `MAX_LEN=256`, 3 epochs.

| Setting | Options | Notes |
|---|---|---|
| `DEBUG` | `True` / `False` | end-to-end pipeline check in a few minutes: 500 stratified train rows, first 300 test rows, 1 epoch, forces `xlm-roberta-base` / `fast` / `FULL_TRAIN=False`. **Never submit a DEBUG output.** |
| `MODEL_NAME` | `xlm-roberta-large` / `xlm-roberta-base` | base is the approved quick fallback |
| `MODE` | `fast` / `kfold` | fast takes about 20–30 min for large on 2×T4; 5-fold large takes about 5× longer |
| `FULL_TRAIN` | `True` / `False` | fast mode only: retrain on 100% of train with the best epoch count and the tuned biases |
| `MAX_LEN` | 256 / 512 | at 512 the per-device batch is halved and grad accumulation doubled, so the effective batch stays the same |

In [ ]:
# ================================ CONFIG: the only cell you need to edit ================================
DEBUG        = False                 # True = quick pipeline check (500 train / 300 test rows, base model, 1 epoch). DO NOT SUBMIT.
MODEL_NAME   = "xlm-roberta-base"  # "xlm-roberta-large" | "xlm-roberta-base" (both approved)
MODE         = "fast"                # "fast" = one 10% stratified holdout | "kfold" = 5-fold stratified CV
FULL_TRAIN   = False                 # fast mode only: after tuning on the holdout, retrain on 100% of train
MAX_LEN      = 256                   # 256 (default) or 512
EPOCHS       = 3
LR           = None                  # None -> 1e-5 for large, 2e-5 for base
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED         = 42
N_FOLDS      = 5                     # kfold mode
HOLDOUT_FRAC = 0.10                  # fast mode
HEAD_FRAC    = 0.60                  # head+tail truncation: keep first 60% / last 40% of the token budget
SUBMISSION_NAME = "Taylor_MYS_ShuijiaoT07_C1.csv"

# ---------------------------- fixed competition constants (do not edit) ----------------------------
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
LANGS = ["pt", "fr", "it", "en"]
EXPECTED_TEST_ROWS = 6000
DEBUG_TRAIN_ROWS, DEBUG_TEST_ROWS = 500, 300

if DEBUG:   # forced settings for a fast end-to-end check
    MODEL_NAME, MODE, FULL_TRAIN, EPOCHS, LR = "xlm-roberta-base", "fast", False, 1, None
    DEBUG_BANNER = "\n" + "!" * 100 + "\n!!!   DEBUG RUN - DO NOT SUBMIT   (500 train rows / 300 test rows / xlm-roberta-base / 1 epoch)\n" + "!" * 100
    print(DEBUG_BANNER)

# ---------------------------- derived per-model defaults (fit a 16 GB T4 in fp16) -------------------
_MODEL_DEFAULTS = {
    "xlm-roberta-large": dict(lr=1e-5, batch=8,  grad_accum=4, eval_batch=64),
    "xlm-roberta-base":  dict(lr=2e-5, batch=16, grad_accum=2, eval_batch=128),
}
assert MODEL_NAME in _MODEL_DEFAULTS, f"MODEL_NAME must be one of {list(_MODEL_DEFAULTS)} (approved backbones only)"
assert MODE in ("fast", "kfold"), "MODE must be 'fast' or 'kfold'"
assert 16 <= MAX_LEN <= 512, "MAX_LEN must be <= 512 (XLM-R position limit)"
_d = _MODEL_DEFAULTS[MODEL_NAME]
LR = LR if LR is not None else _d["lr"]
TRAIN_BATCH, GRAD_ACCUM, EVAL_BATCH = _d["batch"], _d["grad_accum"], _d["eval_batch"]
if MAX_LEN > 256:  # same effective batch, half the activation memory per step
    TRAIN_BATCH, GRAD_ACCUM, EVAL_BATCH = max(1, TRAIN_BATCH // 2), GRAD_ACCUM * 2, max(8, EVAL_BATCH // 2)
HF_MODEL_ID = f"FacebookAI/{MODEL_NAME}"

_banner = f"Model: {HF_MODEL_ID} | debug={DEBUG} | mode={MODE} | full_train={FULL_TRAIN} | max_len={MAX_LEN} | epochs={EPOCHS} | lr={LR}"
try:
    from IPython.display import Markdown, display
    display(Markdown(("## ⚠️ DEBUG RUN - DO NOT SUBMIT\n" if DEBUG else "") + f"### Model: {HF_MODEL_ID}\n`mode={MODE}` · `full_train={FULL_TRAIN}` · `max_len={MAX_LEN}` · "
                     f"`epochs={EPOCHS}` · `lr={LR}` · `batch={TRAIN_BATCH}x{GRAD_ACCUM} accum`"))
except ImportError:
    pass
print(_banner)
print(f"train batch/device={TRAIN_BATCH}, grad_accum={GRAD_ACCUM}, eval batch/device={EVAL_BATCH}")
if MODE == "kfold" and FULL_TRAIN:
    print("NOTE: FULL_TRAIN is ignored in kfold mode (test = average of the fold models).")

## 1 · Environment check
Fails early with a clear message if a required package is missing (internet is off, so nothing gets installed). Prints GPU and library versions, sets seeds everywhere, and disables wandb and other loggers.

In [ ]:
import os, sys, gc, glob, math, time, random, shutil, inspect, importlib, warnings

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

_missing = []
for _pkg in ["numpy", "pandas", "sklearn", "torch", "transformers", "accelerate"]:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    raise ImportError(f"Missing required packages: {_missing}. Internet is OFF and this notebook does not pip install. "
                      "Use the default Kaggle Python GPU image (Settings -> Environment -> 'Latest').")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          TrainerCallback, DataCollatorWithPadding, set_seed)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

transformers.logging.set_verbosity_error()
pd.set_option("display.width", 160)

print(f"python       {sys.version.split()[0]}")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
USE_CUDA = torch.cuda.is_available()
N_GPU = torch.cuda.device_count() if USE_CUDA else 0
USE_FP16 = USE_CUDA
print(f"GPUs         {N_GPU}")
for _i in range(N_GPU):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  cuda:{_i}  {_p.name}  {_p.total_memory / 1024**3:.1f} GB")
if not USE_CUDA:
    print("!" * 90 + "\n!!  NO GPU DETECTED: enable 'GPU T4 x2' under Settings -> Accelerator. Running on CPU (very slow).\n" + "!" * 90)

def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # cheap determinism
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(SEED)

T_START = time.time()
STAGE_TIMES = {}

class Stage:
    """Context manager that prints the elapsed time for each pipeline stage."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t = time.time()
        print(f"\n>>> [{self.name}] started")
        return self
    def __exit__(self, *exc):
        el = time.time() - self.t
        STAGE_TIMES[self.name] = STAGE_TIMES.get(self.name, 0.0) + el
        print(f"<<< [{self.name}] {el / 60:.2f} min  (notebook total {(time.time() - T_START) / 60:.2f} min)")
        return False

## 2 · Locate data and model (no hard-coded paths)
- **Data:** `/kaggle/input/competitions/rmit-hackathon-2026`, falling back to `/kaggle/input/rmit-hackathon-2026`, then any folder under `/kaggle/input` that holds both `train.csv` and `test.csv`.
- **Model:** globs `/kaggle/input/**/config.json` and keeps folders whose path contains the model name (e.g. `xlm-roberta-large`) and that also hold PyTorch weights. If nothing matches, the error names any *other* approved model that is attached and lists everything under `/kaggle/input`. The data is loaded first, so the EDA still works if the model is missing.
- `id` is read as a string so ids with leading zeros (e.g. `t001974`) survive unchanged into the submission.

In [ ]:
INPUT_ROOT = "/kaggle/input"
WORK_DIR   = "/kaggle/working"
CKPT_ROOT  = "/tmp/c1_checkpoints"   # outside /kaggle/working: no stray artifacts

def list_tree(root, max_items=300):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = sorted(d for d in dns if not d.startswith("."))   # skip .cache etc.
        for n in sorted(f for f in fns if not f.startswith(".")):
            out.append(os.path.join(dp, n))
            if len(out) >= max_items:
                return "\n".join(out) + f"\n... (truncated at {max_items})"
    return "\n".join(out) if out else "(empty)"

def find_data_dir():
    for c in [os.path.join(INPUT_ROOT, "competitions", "rmit-hackathon-2026"), os.path.join(INPUT_ROOT, "rmit-hackathon-2026")]:
        if os.path.isfile(os.path.join(c, "train.csv")) and os.path.isfile(os.path.join(c, "test.csv")):
            return c
    hits = sorted(os.path.dirname(p) for p in glob.glob(os.path.join(INPUT_ROOT, "**", "train.csv"), recursive=True)
                  if os.path.isfile(os.path.join(os.path.dirname(p), "test.csv")))
    if hits:
        print(f"WARNING: competition data not at the expected path; using {hits[0]}")
        return hits[0]
    raise FileNotFoundError(f"train.csv/test.csv not found under {INPUT_ROOT}. Attach the competition data. "
                            f"Everything under {INPUT_ROOT}:\n{list_tree(INPUT_ROOT)}")

_WEIGHT_FILES = ["model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"]

def find_model_dir(model_name):
    key = model_name.lower()
    cfgs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "config.json"), recursive=True))
    matches = [os.path.dirname(c) for c in cfgs if key in c.lower().replace("_", "-")]
    usable = [d for d in matches if any(os.path.isfile(os.path.join(d, w)) for w in _WEIGHT_FILES)]
    print(f"config.json files under {INPUT_ROOT}: {len(cfgs)} | path matches '{model_name}': {len(matches)} | with weights: {len(usable)}")
    for d in usable:
        print("   candidate:", d)
    if not usable:
        others = sorted({m for m in _MODEL_DEFAULTS if m != model_name for c in cfgs
                         if m in c.lower().replace("_", "-")})
        hint = (f"\n>>> Found attached: {others}. Either attach 'aisuko/{model_name}' (Add Input -> Models), "
                f"or set MODEL_NAME = \"{others[0]}\" in the config cell.") if others else ""
        raise FileNotFoundError(
            f"No model folder (config.json + weights) whose path contains '{model_name}'. "
            f"Attach the Kaggle Model 'aisuko/{model_name}' (Add Input -> Models).{hint}\n"
            f"Everything under {INPUT_ROOT} (hidden folders like .cache skipped):\n{list_tree(INPUT_ROOT)}")
    usable.sort(key=lambda d: (len(d), d))   # the shortest path is the canonical version folder
    return usable[0]

with Stage("load data"):
    DATA_DIR = find_data_dir()
    print("DATA_DIR  :", DATA_DIR)
    train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), dtype={"id": str})
    test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),  dtype={"id": str})
    for _df in (train_df, test_df):
        _df["text"] = _df["text"].fillna("").astype(str)
        _df["language"] = _df["language"].fillna("unk").astype(str)
    assert set(train_df["label"].unique()) <= set(LABELS), f"unexpected labels: {set(train_df['label'].unique()) - set(LABELS)}"
    if DEBUG:
        _strata = train_df["label"].astype(str) + "_" + train_df["language"].astype(str)
        _keep, _ = train_test_split(np.arange(len(train_df)), train_size=DEBUG_TRAIN_ROWS, stratify=_strata, random_state=SEED)
        train_df = train_df.iloc[np.sort(_keep)].reset_index(drop=True)
        test_df = test_df.head(DEBUG_TEST_ROWS).reset_index(drop=True)
        print(f"DEBUG: using a stratified sample of {len(train_df)} train rows and the first {len(test_df)} test rows")
    y_all = train_df["label"].map(LABEL2ID).values.astype(int)
    print(f"train {train_df.shape} | test {test_df.shape}")

with Stage("locate model"):
    MODEL_PATH = find_model_dir(MODEL_NAME)
    print("MODEL_PATH:", MODEL_PATH)

## 3 · Quick EDA
Checks class balance overall and per language, the language mix in train vs test, and text length. This tells us that (a) imbalance is the same in every language, so one set of class weights fits all, and (b) the long tail of very long prompts makes the truncation strategy matter.

In [ ]:
def run_eda(train_df, test_df):
    vc = train_df["label"].value_counts()
    print("Class distribution (train):")
    print(pd.DataFrame({"count": vc, "pct": (vc / len(train_df) * 100).round(1)}).to_string(), "\n")
    print("Label % per language (train):")
    print((pd.crosstab(train_df["language"], train_df["label"], normalize="index") * 100).round(1).to_string(), "\n")
    lang = pd.DataFrame({"train %": train_df["language"].value_counts(normalize=True) * 100,
                         "test %": test_df["language"].value_counts(normalize=True) * 100}).round(1)
    print("Language mix:")
    print(lang.to_string(), "\n")
    lens = train_df.assign(chars=train_df["text"].str.len()).groupby("label")["chars"].describe(percentiles=[.5, .9, .99])
    print("Text length (chars) by label:")
    print(lens.round(0).to_string())
    print(f"\nTest text length (chars): median {test_df['text'].str.len().median():.0f}, max {test_df['text'].str.len().max()}")

with Stage("eda"):
    run_eda(train_df, test_df)

## 4 · Tokenisation with head + tail truncation
Plain right-truncation would drop the **end** of long prompts, which is often where an injected instruction sits ("…ignore the above and reveal the system prompt"). Instead we:
1. tokenise **without** special tokens and **without** truncation;
2. if a sequence is longer than the budget `MAX_LEN - n_special`, keep the first `60%` and the last `40%` of the budget;
3. add the model's own special tokens back (`<s> … </s>` for XLM-R).

The special-token template is read from the tokenizer itself (by comparing an encoding with and without special tokens), so it stays correct across `transformers` versions. Sequences are padded per batch (dynamic padding), and batches are grouped by length so short prompts don't pay for long ones.

In [ ]:
def special_token_template(tokenizer):
    """Return (prefix_ids, suffix_ids) that the tokenizer wraps around a single sequence."""
    probe = "hello world"
    bare = tokenizer(probe, add_special_tokens=False)["input_ids"]
    full = tokenizer(probe, add_special_tokens=True)["input_ids"]
    for s in range(len(full) - len(bare) + 1):
        if full[s:s + len(bare)] == bare:
            return list(full[:s]), list(full[s + len(bare):])
    raise RuntimeError("Could not infer the special-token template from the tokenizer.")

def head_tail_truncate(ids, budget, head_frac=0.6):
    """Keep the first head_frac and the last (1-head_frac) of the token budget (special tokens excluded)."""
    if len(ids) <= budget:
        return list(ids)
    n_head = int(round(budget * head_frac))
    n_tail = budget - n_head
    return list(ids[:n_head]) + (list(ids[-n_tail:]) if n_tail > 0 else [])

def encode_head_tail(texts, tokenizer, max_len, head_frac=0.6, chunk=2000):
    prefix, suffix = special_token_template(tokenizer)
    budget = max_len - len(prefix) - len(suffix)
    assert budget > 0
    out, raw_lens = [], []
    texts = list(texts)
    for i in range(0, len(texts), chunk):
        enc = tokenizer(texts[i:i + chunk], add_special_tokens=False, truncation=False)["input_ids"]
        for ids in enc:
            raw_lens.append(len(ids))
            out.append(prefix + head_tail_truncate(ids, budget, head_frac) + suffix)
    return out, np.array(raw_lens)

with Stage("tokenize"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    _pre, _suf = special_token_template(tokenizer)
    print(f"tokenizer: {type(tokenizer).__name__} | special template: {tokenizer.convert_ids_to_tokens(_pre)} ... "
          f"{tokenizer.convert_ids_to_tokens(_suf)} | budget = {MAX_LEN - len(_pre) - len(_suf)} content tokens")
    train_ids, train_raw_len = encode_head_tail(train_df["text"], tokenizer, MAX_LEN, HEAD_FRAC)
    test_ids,  test_raw_len  = encode_head_tail(test_df["text"],  tokenizer, MAX_LEN, HEAD_FRAC)
    _budget = MAX_LEN - len(_pre) - len(_suf)
    for _n, _l in [("train", train_raw_len), ("test", test_raw_len)]:
        print(f"{_n}: tokens p50={np.percentile(_l, 50):.0f} p90={np.percentile(_l, 90):.0f} p99={np.percentile(_l, 99):.0f} "
              f"max={_l.max()} | truncated (> {_budget}): {(_l > _budget).mean() * 100:.1f}%")
    assert max(len(x) for x in train_ids + test_ids) <= MAX_LEN

## 5 · Metric, threshold (bias) tuning, reporting and validator helpers
**Why threshold tuning?** Argmax over softmax is the best rule for *accuracy*, not for *macro-F1*. With a 7% minority class, macro-F1 usually improves when the decision boundary moves toward the rare classes. We add a per-class bias `b` to the log-probabilities (`BENIGN` fixed at 0, since only differences matter) and search a grid over `[-2, 2]` (step 0.1) for `JAILBREAK` and `PROMPT_INJECTION`.

**Guarding against overfitting** (the private LB is 70% of test): the macro-F1 surface is **smoothed with a 3×3 mean filter** before taking the argmax, so we pick the centre of a broad plateau rather than a lucky spike. Ties go to the smallest biases, and if tuning doesn't beat plain argmax the biases stay at 0.

In [ ]:
def macro_f1_np(y_true, y_pred, k=3):
    """Fast macro-F1 (identical to sklearn f1_score(average='macro', labels=range(k), zero_division=0))."""
    cm = np.bincount(np.asarray(y_true) * k + np.asarray(y_pred), minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    denom = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    return float(np.mean(np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)))

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def tune_biases(logp, y, grid=None, smooth=True):
    """Grid-search additive biases [0, b_jb, b_pi] on log-probabilities to maximise macro-F1.
    Returns (biases, macro_f1_before, macro_f1_after)."""
    if grid is None:
        grid = np.round(np.arange(-2.0, 2.0 + 1e-9, 0.1), 2)
    grid = np.asarray(grid, dtype=float)
    G = len(grid)
    S = np.zeros((G, G))
    for i, bj in enumerate(grid):
        for j, bp in enumerate(grid):
            S[i, j] = macro_f1_np(y, np.argmax(logp + np.array([0.0, bj, bp]), 1))
    base = macro_f1_np(y, np.argmax(logp, 1))
    if smooth and G >= 3:
        P = np.pad(S, 1, mode="edge")
        S_sel = sum(P[1 + di:1 + di + G, 1 + dj:1 + dj + G] for di in (-1, 0, 1) for dj in (-1, 0, 1)) / 9.0
    else:
        S_sel = S
    score = S_sel - 1e-6 * (grid[:, None] ** 2 + grid[None, :] ** 2)   # tie-break toward small biases
    i, j = np.unravel_index(np.argmax(score), score.shape)
    biases, tuned = np.array([0.0, grid[i], grid[j]]), float(S[i, j])
    if tuned <= base:
        biases, tuned = np.zeros(3), base
    return biases, base, tuned

def eval_report(y, pred, langs, title):
    """classification_report + confusion matrix + macro-F1 per language. Returns {lang: macro_f1, 'ALL': ...}."""
    print("=" * 100 + f"\n{title}\n" + "=" * 100)
    print(classification_report(y, pred, labels=[0, 1, 2], target_names=LABELS, digits=4, zero_division=0))
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1, 2]),
                      index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])
    print(cm.to_string(), "\n")
    res = {"ALL": f1_score(y, pred, labels=[0, 1, 2], average="macro", zero_division=0)}
    langs = np.asarray(langs)
    for lg in LANGS + sorted(set(langs) - set(LANGS)):
        m = langs == lg
        if m.any():
            res[lg] = f1_score(y[m], pred[m], labels=[0, 1, 2], average="macro", zero_division=0)
    print("macro-F1 per language: " + " | ".join(f"{k}={v:.4f}" for k, v in res.items()))
    return res

def validate_submission(sub, test_df, expected_rows=EXPECTED_TEST_ROWS, debug=False):
    """Strict checks: raises AssertionError with a clear message on any problem."""
    assert list(sub.columns) == ["id", "label"], f"columns must be exactly ['id','label'], got {list(sub.columns)}"
    if not debug:
        assert len(sub) == expected_rows, f"expected {expected_rows} rows, got {len(sub)}"
    assert len(sub) == len(test_df), f"row count {len(sub)} != test.csv rows {len(test_df)}"
    assert not sub.isna().any().any(), "submission contains NaN"
    assert sub["id"].is_unique, "duplicate ids in submission"
    assert (sub["id"].astype(str).values == test_df["id"].astype(str).values).all(), "ids do not match test.csv (same order)"
    bad = set(sub["label"].unique()) - set(LABELS)
    assert not bad, f"invalid labels: {bad}"
    return True

## 6 · Model and training setup
- **`AutoModelForSequenceClassification`** with `num_labels=3` and `id2label`/`label2id` set, so the saved config is self-describing.
- **Class-weighted cross-entropy** with weights ∝ `1/sqrt(class frequency)` (normalised to mean 1). Full inverse frequency (about 9× on `PROMPT_INJECTION`) tends to destabilise XLM-R-large and over-predict the rare class. The square root gives the minority classes a meaningful boost, and the bias tuning in section 5 does the final calibration.
- **fp16** mixed precision, AdamW, **linear warmup (10%) then linear decay**, gradient clipping at 1.0.
- Evaluate **every epoch on macro-F1** and keep the best checkpoint (`load_best_model_at_end`, `save_total_limit=1`, weights only, checkpoints in `/tmp` so `/kaggle/working` stays clean).
- **Divergence guard:** XLM-R-large occasionally collapses to the majority class in fp16. If macro-F1 after epoch 1 is below 0.4, a loud warning is printed.
- `TrainingArguments` keys are filtered against the installed `transformers` version, so the notebook runs on both 4.x and 5.x.

In [ ]:
class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, labels=None):
        self.input_ids, self.labels = input_ids, labels
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i], "attention_mask": [1] * len(self.input_ids[i])}
        if self.labels is not None:
            item["labels"] = int(self.labels[i])
        return item

def inv_sqrt_class_weights(y, k=3):
    counts = np.bincount(y, minlength=k).astype(float)
    w = 1.0 / np.sqrt(np.maximum(counts, 1.0))
    return w / w.mean()

class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        self.model_accepts_loss_kwargs = False   # Trainer then scales our mean loss by grad-accum correctly

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs[0]
        loss = F.cross_entropy(logits.float(), labels, weight=self.class_weights.to(logits.device))
        return (loss, outputs) if return_outputs else loss

class DivergenceWatch(TrainerCallback):
    def __init__(self, tag):
        self.tag, self.diverged = tag, False
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        f1 = (metrics or {}).get("eval_macro_f1")
        ep = state.epoch or 0
        if f1 is not None:
            print(f"[{self.tag}] epoch {ep:.2f}: val macro-F1 = {f1:.4f}")
        if f1 is not None and round(ep) == 1 and f1 < 0.4 and not DEBUG:
            self.diverged = True
            print("\n" + "#" * 100 + "\n" + "#" * 100)
            print(f"###  WARNING [{self.tag}]: val macro-F1 after epoch 1 is {f1:.4f} < 0.40. Training has likely DIVERGED")
            print("###  (common for XLM-R-large in fp16). Fix: set LR = 8e-6 in the config cell, or change SEED, and re-run.")
            print("#" * 100 + "\n" + "#" * 100 + "\n")

def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    pred = np.argmax(logits, axis=-1)
    return {"macro_f1": macro_f1_np(eval_pred.label_ids, pred), "accuracy": float((pred == eval_pred.label_ids).mean())}

_TA_SIG = set(inspect.signature(TrainingArguments.__init__).parameters)
_TR_SIG = set(inspect.signature(Trainer.__init__).parameters)
DATALOADER_WORKERS = 0   # main-process loading: worker processes can deadlock at eval time on Kaggle; data is pre-tokenised so this costs ~nothing

def make_training_args(output_dir, n_train, epochs, do_eval):
    steps_per_epoch = math.ceil(math.ceil(n_train / (TRAIN_BATCH * max(1, N_GPU))) / GRAD_ACCUM)
    warmup_steps = int(math.ceil(steps_per_epoch * epochs * WARMUP_RATIO))
    kw = dict(output_dir=output_dir, per_device_train_batch_size=TRAIN_BATCH, per_device_eval_batch_size=EVAL_BATCH,
              gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, num_train_epochs=epochs,
              warmup_steps=warmup_steps, lr_scheduler_type="linear", weight_decay=WEIGHT_DECAY, max_grad_norm=1.0,
              optim="adamw_torch", fp16=USE_FP16, seed=SEED, data_seed=SEED, logging_strategy="steps",
              logging_steps=50, report_to="none", group_by_length=True, dataloader_num_workers=DATALOADER_WORKERS,
              save_only_model=True, save_safetensors=True)
    strat_key = "eval_strategy" if "eval_strategy" in _TA_SIG else "evaluation_strategy"
    if do_eval:
        kw.update({strat_key: "epoch", "save_strategy": "epoch", "load_best_model_at_end": True,
                   "metric_for_best_model": "macro_f1", "greater_is_better": True, "save_total_limit": 1})
    else:
        kw.update({strat_key: "no", "save_strategy": "no"})
    dropped = sorted(k for k in kw if k not in _TA_SIG)
    for crit in [strat_key, "load_best_model_at_end", "fp16", "warmup_steps"]:
        assert crit not in dropped, f"TrainingArguments in this transformers version lacks '{crit}'"
    if dropped:
        print("  (TrainingArguments: unsupported in this version, skipped:", dropped, ")")
    print(f"  steps/epoch={steps_per_epoch} total={steps_per_epoch * epochs} warmup={warmup_steps} "
          f"effective batch={TRAIN_BATCH * max(1, N_GPU) * GRAD_ACCUM}")
    return TrainingArguments(**{k: v for k, v in kw.items() if k in _TA_SIG})

def make_trainer(model, args, train_ds, eval_ds, class_weights, callbacks):
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
              compute_metrics=compute_metrics if eval_ds is not None else None, callbacks=callbacks)
    if "processing_class" in _TR_SIG:
        kw["processing_class"] = tokenizer
    elif "tokenizer" in _TR_SIG:
        kw["tokenizer"] = tokenizer
    return WeightedTrainer(class_weights=class_weights, **kw)

def predict_logits(trainer, input_ids):
    """Predict in length-sorted order (fast with dynamic padding), then restore the original order."""
    order = np.argsort([len(x) for x in input_ids], kind="stable")
    out = trainer.predict(EncodedDataset([input_ids[i] for i in order])).predictions
    out = out[0] if isinstance(out, tuple) else out
    logits = np.empty((len(input_ids), out.shape[1]), dtype=np.float32)
    logits[order] = out.astype(np.float32)
    return logits

def train_and_predict(tr_idx, va_idx, epochs, tag, predict_test=True):
    """Fine-tune on train rows tr_idx (eval on va_idx if given). Returns dict with val/test logits and best epoch."""
    seed_everything(SEED)
    out_dir = os.path.join(CKPT_ROOT, tag)
    shutil.rmtree(out_dir, ignore_errors=True)
    cw = inv_sqrt_class_weights(y_all[tr_idx])
    print(f"[{tag}] train={len(tr_idx)} val={0 if va_idx is None else len(va_idx)} epochs={epochs} "
          f"class weights={dict(zip(LABELS, [round(float(w), 3) for w in cw]))}")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=len(LABELS),
                                                               id2label=ID2LABEL, label2id=LABEL2ID)
    train_ds = EncodedDataset([train_ids[i] for i in tr_idx], y_all[tr_idx])
    eval_ds = None
    if va_idx is not None:
        va_sorted = sorted(va_idx, key=lambda i: len(train_ids[i]))   # faster per-epoch eval
        eval_ds = EncodedDataset([train_ids[i] for i in va_sorted], y_all[va_sorted])
    watch = DivergenceWatch(tag)
    args = make_training_args(out_dir, len(tr_idx), epochs, do_eval=eval_ds is not None)
    trainer = make_trainer(model, args, train_ds, eval_ds, cw, [watch])
    trainer.train()

    res = {"best_epoch": epochs, "diverged": watch.diverged, "val_logits": None, "test_logits": None, "history": None}
    if eval_ds is not None:
        hist = [h for h in trainer.state.log_history if "eval_macro_f1" in h]
        res["history"] = pd.DataFrame([{"epoch": round(h["epoch"], 2), "eval_loss": h.get("eval_loss"),
                                        "eval_macro_f1": h["eval_macro_f1"], "eval_accuracy": h.get("eval_accuracy")} for h in hist])
        best = max(hist, key=lambda h: h["eval_macro_f1"])
        res["best_epoch"] = max(1, int(round(best["epoch"])))
        print(f"[{tag}] per-epoch validation:\n{res['history'].to_string(index=False)}")
        print(f"[{tag}] best epoch = {res['best_epoch']} (macro-F1 {best['eval_macro_f1']:.4f}), best checkpoint loaded")
        res["val_logits"] = predict_logits(trainer, [train_ids[i] for i in va_idx])
    if predict_test:
        res["test_logits"] = predict_logits(trainer, test_ids)

    # free GPU memory before the next fold / full-train model
    del trainer, model, train_ds, eval_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return res

## 7 · Train: holdout (`fast`) or 5-fold CV (`kfold`)
Splits are stratified on the **label × language** combination, so every fold has the same class mix in every language. That makes per-language validation scores comparable to the test set.
- `fast`: one 10% holdout. With `FULL_TRAIN=True`, a second model is then trained on **100%** of train for the best epoch count found on the holdout, and it predicts test. Bias tuning still uses the holdout predictions.
- `kfold`: 5 models. OOF predictions cover all of train (a more reliable basis for bias tuning), and the test prediction is the **average of the 5 fold softmax probabilities**.

In [ ]:
strata = (train_df["label"].astype(str) + "_" + train_df["language"].astype(str)).values
all_idx = np.arange(len(train_df))
if MODE == "fast":
    _tr, _va = train_test_split(all_idx, test_size=HOLDOUT_FRAC, stratify=strata, random_state=SEED, shuffle=True)
    SPLITS = [(np.sort(_tr), np.sort(_va))]
else:
    SPLITS = [(tr, va) for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(all_idx, strata)]

oof_probs = np.full((len(train_df), len(LABELS)), np.nan)
test_prob_list, best_epochs, diverged_any = [], [], False

for f, (tr_idx, va_idx) in enumerate(SPLITS):
    tag = "holdout" if MODE == "fast" else f"fold{f + 1}of{len(SPLITS)}"
    with Stage(f"train {tag}"):
        res = train_and_predict(tr_idx, va_idx, EPOCHS, tag, predict_test=not (MODE == "fast" and FULL_TRAIN))
    oof_probs[va_idx] = softmax_np(res["val_logits"])
    best_epochs.append(res["best_epoch"])
    diverged_any |= res["diverged"]
    if res["test_logits"] is not None:
        test_prob_list.append(softmax_np(res["test_logits"]))
    print(f"[{tag}] macro-F1 (argmax) = {macro_f1_np(y_all[va_idx], oof_probs[va_idx].argmax(1)):.4f}")

if MODE == "fast" and FULL_TRAIN:
    full_epochs = best_epochs[0]
    print(f"\nFULL_TRAIN: retraining on 100% of train ({len(all_idx)} rows) for {full_epochs} epoch(s) (best holdout epoch)")
    with Stage("train full"):
        res = train_and_predict(all_idx, None, full_epochs, "full", predict_test=True)
    test_prob_list = [softmax_np(res["test_logits"])]

test_probs = np.mean(test_prob_list, axis=0)
print(f"\nbest epochs per split: {best_epochs} | test probability sets averaged: {len(test_prob_list)}")

## 8 · Threshold (bias) tuning and validation report
Biases are tuned on the **validation / OOF probabilities only** and then frozen. We report the full classification report, confusion matrix and **macro-F1 per language** before and after, so we can see whether tuning helps every language and doesn't just trade one language off against another.

In [ ]:
with Stage("bias tuning + report"):
    vmask = ~np.isnan(oof_probs).any(1)
    v_y, v_logp, v_lang = y_all[vmask], np.log(oof_probs[vmask] + 1e-12), train_df["language"].values[vmask]
    BIASES, f1_before, f1_after = tune_biases(v_logp, v_y)
    print(f"Validation rows used: {vmask.sum()}  ({'OOF over all folds' if MODE == 'kfold' else 'holdout'})")
    print(f"Chosen biases (added to log-probs): " + ", ".join(f"{l}={b:+.2f}" for l, b in zip(LABELS, BIASES)))
    print(f"Val macro-F1  before tuning: {f1_before:.4f}   after tuning: {f1_after:.4f}   (delta {f1_after - f1_before:+.4f})\n")

    rep_before = eval_report(v_y, v_logp.argmax(1), v_lang, "VALIDATION: BEFORE threshold tuning (argmax)")
    rep_after  = eval_report(v_y, (v_logp + BIASES).argmax(1), v_lang, f"VALIDATION: AFTER threshold tuning (biases {BIASES.round(2).tolist()})")
    cmp = pd.DataFrame({"before": rep_before, "after": rep_after}).T.round(4)
    print("\nmacro-F1 summary (ALL + per language):")
    print(cmp.to_string())
    if (diverged_any or f1_before < 0.4) and not DEBUG:
        print("\n" + "#" * 100 + "\n###  WARNING: a run diverged or val macro-F1 is < 0.40. Lower LR (e.g. 8e-6) or change SEED before submitting.\n" + "#" * 100)

## 9 · Test inference and submission
The frozen biases are applied to the test log-probabilities (averaged over folds in `kfold` mode). Only `/kaggle/working/Taylor_MYS_ShuijiaoT07_C1.csv` is written: no probability files or other artifacts.

In [ ]:
with Stage("write submission"):
    test_pred = (np.log(test_probs + 1e-12) + BIASES).argmax(1)
    submission = pd.DataFrame({"id": test_df["id"].astype(str).values, "label": [ID2LABEL[int(i)] for i in test_pred]})
    os.makedirs(WORK_DIR, exist_ok=True)
    SUB_PATH = os.path.join(WORK_DIR, SUBMISSION_NAME)
    submission.to_csv(SUB_PATH, index=False)
    print(f"wrote {SUB_PATH}  ({len(submission)} rows)")
    print(submission.head())

## 10 · Submission validator
Re-reads the CSV **from disk** and checks: exactly 6,000 rows, columns exactly `["id","label"]`, ids identical to `test.csv` in the same order, no NaN, and only the three allowed labels. It also compares the predicted label mix with the train mix as a sanity check (a big shift, e.g. 0% `PROMPT_INJECTION`, means something is wrong).

In [ ]:
_chk = pd.read_csv(SUB_PATH, dtype={"id": str})
validate_submission(_chk, test_df, debug=DEBUG)
dist = pd.DataFrame({"train %": train_df["label"].value_counts(normalize=True) * 100,
                     "test pred %": _chk["label"].value_counts(normalize=True) * 100}).reindex(LABELS).fillna(0).round(2)
print("SUBMISSION VALID:", SUB_PATH)
print(dist.to_string())
print("\nStage timings (min):", {k: round(v / 60, 2) for k, v in STAGE_TIMES.items()},
      f"| total {(time.time() - T_START) / 60:.1f} min")
print(f"\nSummary: {HF_MODEL_ID} | mode={MODE} | full_train={FULL_TRAIN} | val macro-F1 {f1_before:.4f} -> {f1_after:.4f} "
      f"| biases {dict(zip(LABELS, [round(float(b), 2) for b in BIASES]))}")
if DEBUG:
    print(DEBUG_BANNER + f"\n!!!   {SUB_PATH} has only {len(_chk)} rows: it is a pipeline check, not a submission.\n" + "!" * 100)

## 11 · Results & Discussion

| # | Model | Mode | FULL_TRAIN | MAX_LEN | Val macro-F1 (argmax) | Val macro-F1 (tuned) | Biases (JB, PI) | Public LB |
|---|---|---|---|---|---|---|---|---|
| 1 | xlm-roberta-base | fast | False | 256 | 0.9373 | 0.9397 | 0.0, +0.7 | _fill after submission_ |

**Run details:** 2×T4, 19.0 min end-to-end (18.8 min training, 3 epochs). Validation = 1,400-row holdout stratified on label × language.

**Per-class (validation, after tuning)**

| Class | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| BENIGN | 0.9532 | 0.9609 | 0.9570 | 869 |
| JAILBREAK | 0.9279 | 0.9215 | 0.9247 | 433 |
| PROMPT_INJECTION | 0.9574 | 0.9184 | 0.9375 | 98 |
| **Macro avg** | 0.9462 | 0.9336 | **0.9397** | 1400 |

**Per-language macro-F1 (validation)**

| | pt | fr | it | en |
|---|---|---|---|---|
| before tuning | 0.9160 | 0.9565 | 0.9682 | 0.9164 |
| after tuning | 0.9160 | 0.9565 | 0.9682 | 0.9466 |

### Discussion
- **Strong multilingual baseline.** One fine-tuned XLM-R-base model reaches 0.940 macro-F1 on held-out data across pt/fr/it/en (an all-BENIGN baseline scores about 0.26). It uses no translation and no language-specific models.
- **The rare class is handled well.** With inverse-sqrt class weights, PROMPT_INJECTION (7% of data) reaches F1 0.938, on par with the majority class. The predicted test label mix (62.7 / 30.7 / 6.7%) matches the training prior (62 / 31 / 7%), so the model neither ignores nor over-predicts it.
- **Bias tuning was a small, targeted correction.** The search only boosted PROMPT_INJECTION (+0.7 on its log-probability). It recovered 2 more injections (recall 0.898 → 0.918) at the cost of 1 extra false alarm, for +0.0024 macro-F1. The whole gain came from English (0.916 → 0.947); pt/fr/it were unchanged. The English holdout is small (about 10% of 1,400 rows), so we treat this as a modest, low-confidence gain. The smoothed grid search is designed to avoid fitting noise.
- **The main remaining error is BENIGN ↔ JAILBREAK, not the rare class.** 64 of the 76 validation errors are BENIGN↔JAILBREAK confusions (30 + 34). Only 8 injections are missed. Role-play and "pretend" style prompts sit close to that boundary, so this is where further gains are.
- **Portuguese is the weakest language (0.916)** even though it is 45% of the data, while French and Italian exceed 0.95. Since pt is also the largest share of the test set, it is the most valuable language to improve.
- **Model choice.** We used the approved `xlm-roberta-base` because the `xlm-roberta-large` Kaggle Model repeatedly failed to mount (`ERRORED_MOUNTING_DATASET`). The pipeline is model-agnostic: switching is a one-line config change.
- **Generalisation.** We choose submissions by validation macro-F1, not by the public leaderboard (only 30% of test), to avoid overfitting the public split.

### What we'd try next
1. **5-fold CV (`MODE="kfold"`)**: average 5 models and tune biases on all 14,000 out-of-fold predictions. This gives about 980 PROMPT_INJECTION examples instead of 98, so the biases are far more reliable.
2. **`FULL_TRAIN=True`**: retrain on 100% of the training data with the best epoch count.
3. **`xlm-roberta-large`** once the Kaggle Model mounts reliably, mainly to sharpen the BENIGN ↔ JAILBREAK boundary.
4. **Error analysis on Portuguese and on BENIGN ↔ JAILBREAK confusions**, to see whether the errors are long prompts (try `MAX_LEN=512`) or genuinely ambiguous role-play.
5. **Seed averaging** (2–3 seeds) for a more stable final prediction.